# 02 Shuffled-Label UdonPred Null Workflow


## 1. Setup

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "UdonPred").exists():
    ROOT = ROOT.parent

DATASETS = ["trizod", "chezod", "softdis", "pdbflex", "atlas", "plddt", "disprot"]
METRIC_COLUMNS = ["trizod", "chezod", "softdis", "pdbflex", "atlas", "plddt", "disprot\n(AP)", "disprot\n(AUROC)"]
RESULT_DIR = ROOT / "results" / "udonpred_shuffled_labels"

print("Python:", sys.version)
print("Project root:", ROOT)
print("Result dir:", RESULT_DIR)

## 2. CUDA Check

In [ ]:
cuda_cmd = ["uv", "run", "python", str(ROOT / "scripts" / "check_gpu.py")]
print(" ".join(cuda_cmd))
subprocess.run(cuda_cmd, cwd=ROOT / "UdonPred", check=True)

## 3. Dataset And Label Sanity Checks

In [ ]:
def read_jsonl(path):
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

rows = []
for dataset in DATASETS:
    dataset_dir = ROOT / "UdonPred" / "data" / dataset
    for split in ["train", "valid", "test"]:
        records = read_jsonl(dataset_dir / f"{split}.jsonl")
        residues = 0
        valid_labels = 0
        masked_labels = 0
        bad_lengths = []
        for record in records:
            sequence = str(record["x_0"])
            labels = record["y"]
            residues += len(sequence)
            valid_labels += sum(1 for value in labels if value != 999)
            masked_labels += sum(1 for value in labels if value == 999)
            if len(sequence) != len(labels):
                bad_lengths.append(record["id"])
        rows.append({
            "dataset": dataset,
            "split": split,
            "records": len(records),
            "residues": residues,
            "valid_labels": valid_labels,
            "masked_labels": masked_labels,
            "bad_lengths": len(bad_lengths),
        })

sanity = pd.DataFrame(rows)
display(sanity)
assert sanity["bad_lengths"].sum() == 0, "Found sequence/label length mismatches"

## 4. Launch Shuffled Null Runs

This is expensive: the default runs 3 seeds x 7 training datasets and exports 21 heads. Test-set ProstT5 embeddings are saved under the result directory and reused across shuffled heads and seeds.

In [ ]:
RUN_NULL_WORKFLOW = True

SEEDS = [10]
DEVICE = "cuda"
BATCH_SIZE = 2000
EMBEDDING_CACHE_DIR = RESULT_DIR / "embeddings"
TRAINING_HF_CACHE_DIR = RESULT_DIR / "training_hf_cache"
# SMOOTH = 1.5

# Optional smoke-run controls. Leave DATASETS as-is for the full workflow.
RUN_DATASETS = DATASETS
NUM_TRAIN_EPOCHS = 20

cmd = [
    "uv", "run", "python",
    str(ROOT / "scripts" / "run_udonpred_shuffled_null.py"),
    "--udonpred-dir", str(ROOT / "UdonPred"),
    "--output-dir", str(RESULT_DIR),
    "--device", DEVICE,
    "--batch-size", str(BATCH_SIZE),
    "--embedding-cache-dir", str(EMBEDDING_CACHE_DIR),
    "--training-hf-cache-dir", str(TRAINING_HF_CACHE_DIR),
    # "--smooth", str(SMOOTH),
    "--train-device", DEVICE,
    "--seeds", *map(str, SEEDS),
    "--datasets", *RUN_DATASETS,
]
if NUM_TRAIN_EPOCHS is not None:
    cmd += ["--num-train-epochs", str(NUM_TRAIN_EPOCHS)]

print(" ".join(cmd))
if RUN_NULL_WORKFLOW:
    subprocess.run(cmd, cwd=ROOT / "UdonPred", check=True)
else:
    print("Set RUN_NULL_WORKFLOW = True to launch the shuffled-label null workflow.")

## 5. Load Null Matrices

In [ ]:
matrix_rows = []
for path in sorted(RESULT_DIR.glob("seed_*/matrix.csv")):
    seed = int(path.parent.name.split("_", 1)[1])
    matrix = pd.read_csv(path)
    matrix["seed"] = seed
    matrix_rows.append(matrix)

if matrix_rows:
    null_matrices = pd.concat(matrix_rows, ignore_index=True)
    display(null_matrices)
else:
    null_matrices = pd.DataFrame(columns=["seed", "train_dataset", *METRIC_COLUMNS])
    print(f"No shuffled null matrices found under {RESULT_DIR}")
    print("Run section 4 with RUN_NULL_WORKFLOW = True, or point RESULT_DIR at an existing shuffled-label output directory.")

## 6. Null Summary

In [ ]:
if null_matrices.empty:
    long_null = pd.DataFrame(columns=["seed", "train_dataset", "test_metric", "score"])
    summary = pd.DataFrame(columns=["train_dataset", "test_metric", "n", "mean", "std", "q025", "q500", "q975"])
    print("No null matrices loaded; skipping summary until matrix.csv files exist.")
else:
    long_null = null_matrices.melt(
        id_vars=["seed", "train_dataset"],
        value_vars=METRIC_COLUMNS,
        var_name="test_metric",
        value_name="score",
    ).dropna()

    summary = (
        long_null.groupby(["train_dataset", "test_metric"])["score"]
        .agg(
            n="count",
            mean="mean",
            std="std",
            q025=lambda s: s.quantile(0.025),
            q500="median",
            q975=lambda s: s.quantile(0.975),
        )
        .reset_index()
    )
    display(summary)

## 7. Heatmaps

In [ ]:
if summary.empty:
    mean_matrix = pd.DataFrame(index=DATASETS, columns=METRIC_COLUMNS, dtype=float)
    print("No null summary available; skipping null heatmap.")
else:
    mean_matrix = summary.pivot(index="train_dataset", columns="test_metric", values="mean").reindex(index=DATASETS, columns=METRIC_COLUMNS)

    plt.figure(figsize=(10, 6))
    sns.heatmap(mean_matrix, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1)
    plt.xlabel("Test dataset / metric")
    plt.ylabel("Shuffled-label training dataset")
    plt.title("Shuffled-label null mean performance")
    plt.tight_layout()
    plt.show()

In [ ]:
observed_path = ROOT / "results" / "udonpred_matrix" / "matrix.csv"
if summary.empty:
    print("No null summary available; skipping observed-minus-null heatmap.")
elif observed_path.exists():
    observed = pd.read_csv(observed_path).set_index("train_dataset").reindex(index=DATASETS, columns=METRIC_COLUMNS)
    delta = observed - mean_matrix
    plt.figure(figsize=(10, 6))
    sns.heatmap(delta, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1)
    plt.xlabel("Test dataset / metric")
    plt.ylabel("Training dataset")
    plt.title("Observed UdonPred matrix minus shuffled-label null mean")
    plt.tight_layout()
    plt.show()
else:
    print(f"Observed matrix not found: {observed_path}")

## 8. Distribution View

In [ ]:
if long_null.empty:
    print("No null scores available; skipping distribution plot.")
else:
    diagonal_metrics = [dataset for dataset in DATASETS if dataset != "disprot"] + ["disprot\n(AP)", "disprot\n(AUROC)"]
    plot_rows = []
    for dataset in DATASETS:
        metric = dataset if dataset != "disprot" else "disprot\n(AP)"
        plot_rows.append(long_null[(long_null["train_dataset"] == dataset) & (long_null["test_metric"] == metric)])
    diagonal_null = pd.concat(plot_rows, ignore_index=True)

    plt.figure(figsize=(10, 6))
    sns.stripplot(data=diagonal_null, x="test_metric", y="score", hue="train_dataset", dodge=False)
    plt.xticks(rotation=45, ha="right")
    plt.xlabel("Same-dataset metric")
    plt.ylabel("Null score")
    plt.title("Shuffled-label same-dataset null scores")
    plt.tight_layout()
    plt.show()